# 📋 [OPS08_M06] 대공 사격 구역 할당 및 인접 분대 상황 전파 인계 실습
### AI Co-pilot 협업 기반 소부대 대공 방어 표준운용절차(SOP) 및 모의훈련 워크북 (정답 & 교관 가이드북)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS08/ops08_m06_workbook.ipynb)

---

## 📌 프로젝트 개요
- **과정명:** 드론 경보 시 관측자-지휘자-경계조 역할분담 모의훈련 (Level 2)
- **미션 번호:** OPS08_M06
- **미션 주제:** 대공 사격 구역 할당 및 인접 분대 상황 전파 인계 실습
- **운용 원칙:**
  1. **HITL 원칙:** AI 제안은 참고 지표이며, 최종 지휘·승인·교전 판단은 인간이 확정한다.
  2. **군사보안 준수:** 실제 부대명, 좌표, 통신 음어, 무기 제원은 일체 사용하지 않으며 가상 시뮬레이션 데이터만 활용한다.
  3. **안전 제일 원칙:** 실사격/장비 조작은 배제하며 구두 선언과 모의 기동으로만 진행하고 교관 지휘에 엄격히 따른다.


In [ ]:
# [OPS08] 소부대 드론 경보 모의훈련 자동 분석 및 검증 헬퍼 모듈
import pandas as pd
import numpy as np

def verify_mission1_raci(raci_data, checklist_items):
    """미션 1 RACI 매트릭스 및 체크리스트 무결성 검증"""
    print("=" * 60)
    print("📋 [미션 1 검증] 7단계 역할분담 매트릭스 & 체크리스트 검증")
    print("=" * 60)
    
    stages = ["포착", "식별", "보고", "전파", "결심", "대피·대응", "해제"]
    stage_pass = True
    for s in stages:
        if s not in raci_data:
            print(f"❌ 누락된 단계: {s}")
            stage_pass = False
        else:
            roles = raci_data[s]
            acc_count = sum(1 for r, duty in roles.items() if "책" in duty or "A" in duty)
            if acc_count != 1:
                print(f"⚠️ [{s} 단계] 책임자(A/책)가 정확히 1명이 아닙니다! (현재 {acc_count}명)")
                stage_pass = False
    
    check_count = len(checklist_items)
    check_pass = check_count >= 15
    print(f"✔ 현장 체크리스트 항목 수: {check_count}개 / 기준 15개 이상 {'[PASS]' if check_pass else '[FAIL]'}")
    print(f"✔ 7단계 RACI 매트릭스 책임자 무결성: {'[PASS]' if stage_pass else '[FAIL]'}")
    
    if stage_pass and check_pass:
        print("\n🎉 [미션 1 PASS] 현장 점검 기준과 역할 분담이 전술 기준에 완벽히 부합합니다!")
    else:
        print("\n❌ [보완 필요] 누락된 단계나 중복된 책임자를 점검하십시오.")

def analyze_mission4_dryrun(log_df):
    """미션 4 1차 모의훈련 타임라인 및 준수율 분석"""
    print("=" * 60)
    print("⏱️ [미션 4 분석] 1차 Dry-run 훈련 타임라인 및 SOP 준수율 계산")
    print("=" * 60)
    
    avg_lead_time = log_df['lead_time_sec'].mean()
    sop_pass_rate = (log_df['sop_pass'].sum() / len(log_df)) * 100
    
    print(log_df[['event_id', 'detect_sec', 'report_sec', 'relay_sec', 'action_sec', 'lead_time_sec', 'sop_pass']])
    print("-" * 60)
    print(f"✔ 평균 조치 완료 시간: {avg_lead_time:.1f}초 (기준: 180초 이내) -> {'[PASS]' if avg_lead_time <= 180 else '[FAIL]'}")
    print(f"✔ SOP 단계별 준수율: {sop_pass_rate:.1f}% (기준: 80% 이상) -> {'[PASS]' if sop_pass_rate >= 80 else '[FAIL]'}")
    
    if avg_lead_time <= 180 and sop_pass_rate >= 80:
        print("\n🎉 [미션 4 PASS] 1차 Dry-run 목표치를 달성하였습니다! 병목 구간 2건을 AAR에 인계하십시오.")

def grade_mission5_cross_verification(user_answers, answer_key):
    """미션 5 교차확인 판정 채점"""
    print("=" * 60)
    print("🎯 [미션 5 채점] 오류 주입 교차확인 판정 결과 검증")
    print("=" * 60)
    
    correct = 0
    miss_threat = 0
    total = len(answer_key)
    
    for card, info in answer_key.items():
        user_pred = user_answers.get(card, {}).get("pred")
        truth = info["truth"]
        is_threat = ("실제 위협" in truth)
        
        is_correct = (user_pred == truth)
        if is_correct:
            correct += 1
        elif is_threat and ("오경보" in str(user_pred)):
            miss_threat += 1
            print(f"🚨 [치명적 오류] {card} 위협을 오경보로 놓쳤습니다! (Miss 발생)")
            
    success_rate = (correct / total) * 100
    print(f"✔ 허위 경보 식별 성공률: {success_rate:.1f}% ({correct}/{total}) (기준: 85% 이상)")
    print(f"✔ 실제 위협 미탐(Miss) 건수: {miss_threat}건 (기준: 0건)")
    
    if success_rate >= 85.0 and miss_threat == 0:
        print("\n🎉 [미션 5 PASS] 교차 확인을 통해 적 기만 및 허위 경보를 완벽히 필터링했습니다!")

def compare_mission9_retrial(m4_data, m9_data):
    """미션 9 1차 대비 2차 훈련 개선율 분석"""
    print("=" * 60)
    print("📈 [미션 9 성과 분석] 1차(M4) 대비 2차 재훈련 반응 속도 & 오류 개선율")
    print("=" * 60)
    
    time1 = m4_data['avg_time']
    time2 = m9_data['avg_time']
    time_reduction = ((time1 - time2) / time1) * 100
    
    err1 = m4_data['errors']
    err2 = m9_data['errors']
    err_reduction = ((err1 - err2) / err1) * 100
    
    print(f"1. 전체 조치 소요 시간: {time1:.1f}초 ➔ {time2:.1f}초 (단축률: {time_reduction:.1f}% / 목표 30% 이상)")
    print(f"2. SOP 절차 오류 건수: {err1}건 ➔ {err2}건 (감소율: {err_reduction:.1f}% / 목표 50% 이상)")
    print("-" * 60)
    
    pass_time = time_reduction >= 30.0
    pass_err = err_reduction >= 50.0
    
    if pass_time and pass_err:
        print("🎉 [미션 9 PASS] 수정 SOP 적용을 통해 팀 단위 전술 반응 속도가 입증되었습니다!")


---

## 🎯 미션 06. 대공 사격 구역 할당 및 인접 분대 상황 전파 인계 실습

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"가상 진지 4대 구역의 사격 구역 중복과 사각지대를 조정한 재할당 표와 SBAR 10개 필수 항목 인계 브리핑 문안을 작성해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 구역 중복·공백이 0건인지, SBAR 브리핑이 10분 내에 전달 가능한 명확한 핵심 정보로 구성되었는지 검증합니다.

---

### 📑 실습자료 및 과업 지침

대공 사격 구역 할당 및 인접 분대 상황 전파 인계 실습**

가. 제공 자료 - 가상 진지 구역표(추상화, 실제 배치 아님) [교육용가상자료]

| **구역** | **방향(진지 정면 12시 기준)** | **지형 특징(가상)** | **관측 난이도** |
| --- | --- | --- | --- |
| **A** | 12시~3시 | 개활지, 시야 넓음 | 낮음 |
| **B** | 3시~6시 | 숲 가장자리, 저고도 접근 시 늦게 보임 | 높음 |
| **C** | 6시~9시 | 후방 도로, 아군 이동 잦음 | 보통(아군 오인 주의) |
| **D** | 9시~12시 | 낮은 능선, 역광 시간대 있음 | 보통 |

나. 제공 자료 - 분대별 감시·사격 구역 초안(오류 포함)

| **분대** | **초안 감시·사격 구역** | **조정안(작성)** | **조정 근거(작성)** |
| --- | --- | --- | --- |
| **1분대** | 11시~3시 |  |  |
| **2분대** | 2시~6시 |  |  |
| **3분대** | 7시~10시 |  |  |

| **중복 구간** | ( ) ※ 초안에서 찾은 겹치는 방향을 모두 적는다 |
| --- | --- |

| **공백 구간** | ( ) ※ 어느 분대도 맡지 않은 방향을 모두 적는다 |
| --- | --- |

| **경계 방향 처리 원칙** | ( ) ※ 예: 경계선 방향은 양쪽 분대가 공동 관측하고 사격은 한쪽만 담당 |
| --- | --- |

다. 제공 자료 - 인계 상황 카드(진행 중 상황) [교육용가상자료]

| 가상 05:55 현재: 드론 경보 발령 중. 05:43에 2시 방향 회전익 1기 접근 후 05:49 1시 방향으로 이탈, 05:52 4시 방향 소리만 청취(확인 요망 상태). 경계조 1명 발목 경상(모의, 응급처치 완료). 무전기 잡음 심함, 06:00 교대 예정. 상급 보고는 05:45 1회 완료, 다음 보고 예정 06:00. |
| --- |

라. 작성 서식 1 - SBAR 인계 브리핑 템플릿

| **구분** | **들어갈 내용** | **작성** |
| --- | --- | --- |
| **S 상황** | 현재 경보 단계, 지금 보고 있는 표적 |  |
| **B 배경** | 지금까지 경과(시각순), 이미 한 조치 |  |
| **A 평가** | 위협 판단, 불확실한 점, 인원·통신 상태 |  |
| **R 요청** | 인수자가 곧바로 할 일, 다음 보고 시각 |  |

마. 작성 서식 2 - 인계 필수 항목 10개 확인표

| **번호** | **필수 항목** | **전달(O/X)** | **인수자 복창(O/X)** |
| --- | --- | --- | --- |
| **1** | 현재 경보 단계 | ( ) | ( ) |
| **2** | 표적 수·방위·마지막 확인 시각 | ( ) | ( ) |
| **3** | 확인 요망 중인 신호 | ( ) | ( ) |
| **4** | 담당 감시·사격 구역 | ( ) | ( ) |
| **5** | 인원 현황·부상자 | ( ) | ( ) |
| **6** | 통신 상태 | ( ) | ( ) |
| **7** | 상급 보고 이력 | ( ) | ( ) |
| **8** | 다음 보고 예정 시각 | ( ) | ( ) |
| **9** | 인수자 즉시 조치 사항 | ( ) | ( ) |
| **10** | 인계·인수 시각 | ( ) | ( ) |

| **인계 소요 시간** | 시작 ( : : ) ~ 종료 ( : : ) = ( )분 → 목표 10분 이내 ( ) |
| --- | --- |

| **누락률** | 누락 항목 수 ( ) ÷ 10 × 100 = ( )% → 목표 0% ( ) |
| --- | --- |

| **상호 서명** | 인계자 ( ) 인수자 ( ) |
| --- | --- |

바. 인수자 돌발 상황 퀴즈 (교관이 인계 직후 구두로 질문)

| **번호** | **질문** | **정답(교관용)** | **O/X** |
| --- | --- | --- | --- |
| **Q1** | 지금 확인 요망 상태인 신호는 어느 방향인가? | 4시 방향(소리만) | ( ) |
| **Q2** | 마지막으로 육안 확인된 드론은 어디로 이탈했나? | 1시 방향 | ( ) |
| **Q3** | 부상자는 몇 명이고 상태는? | 1명, 발목 경상, 응급처치 완료 | ( ) |
| **Q4** | 다음 상급 보고 시각은? | 06:00 | ( ) |
| **Q5** | 지금 4시 방향에서 윙윙 소리가 커지면 가장 먼저 할 일은? | 은폐 전파 후 7항목 보고 | ( ) |

| **점검 항목** | **확인** |
| --- | --- |
| 구역 중복·공백이 0개가 되도록 조정했다 | ( ) |
| 조정 근거에 지형·관측 난이도를 반영했다 | ( ) |
| SBAR 4개 칸을 모두 채웠다 | ( ) |
| 필수 항목 10개 누락 0 | ( ) |
| 인계를 10분 이내에 마쳤다 | ( ) |
| 인수자 퀴즈 4문항 이상 정답 | ( ) |
| 상호 서명을 받았다 | ( ) |

**

